In [47]:
import json
import os
from IPython.display import Image 
from IPython.display import clear_output, display
import pandas as pd
from ipywidgets import HBox, Output

In [48]:
step = 0
subset = 'phase2'
uncertainty = 'se_scores'
experiment = 'cub200_experiment_llama_wo_scaling'

In [49]:
data = json.load(open(f'checkpoints/{experiment}/{subset}_round{step}/selected_samples.json', "r"))
len(data)

100

In [50]:
label_path = f'checkpoints/{experiment}/{subset}_round{step}/cub200/concept_labels.json'

In [51]:
def load_labels(path):
    with open(path, 'r') as f:
        labels = json.load(f)
    
    return labels

def search_keys(pattern, labels) -> str:
    if pattern in labels:
        return pattern
    pattern = pattern[:10]
    matching_keys = [key for key in labels.keys() if pattern in key]
    return matching_keys[0] if matching_keys else None

In [52]:

n_samples = len(data)

for sample in data:
    concept_labels = load_labels(label_path)
    already_labeled = load_labels("checkpoints/cub200_experiment_llama_wo_scaling/reference/reference.json")
        
    image_id = sample['image']
    if image_id in concept_labels:
        n_samples -= 1
        continue  # already labeled

    concept_labels[image_id] = {k: '' for k in sample[uncertainty].keys()}
    image_path = os.path.join('data/cub200/images', image_id)
    if not os.path.exists(image_path):
        print(f"[Warning] Missing image: {image_path}")
        continue

    already_labeled_image = search_keys(image_id, already_labeled)
    if already_labeled_image:
        already_labeled_image_path = os.path.join('data/cub200/images', already_labeled_image)
    if already_labeled_image == image_id:
        concept_labels[image_id] = already_labeled[already_labeled_image]
        with open(label_path, "w") as f:
            json.dump(concept_labels, f, indent=2)
        n_samples -= 1
        continue

    # Use IPython display for Jupyter image rendering
    image = Image(filename=image_path, width=500, height=500)
    labeled_image = Image(filename=already_labeled_image_path, width=300, height=300) if already_labeled_image else None

    for concept in sample[uncertainty]:
        sample[uncertainty][concept] = max(sample[uncertainty][concept], 0.0)  # ensure no negative scores
    sorted_scores = sorted(sample[uncertainty].items(), key=lambda x: x[1], reverse=True)

    for concept, score in sorted_scores:
        # if score > 0.0001 or step == 0 or subset == 'test':
        clear_output(wait=True)  # clear before everything
        # print(f"\n[Info] Processing {image_id} (already labeled: {already_labeled_image})")
        print(f"{n_samples} images left.")
        print(f"Image: {image_id}")
        print(f"[Uncertainty] {concept}: {score:.4f}\n")

        print("Possible concepts:")
        for i, c in enumerate(sample['concepts'][concept]):
            print(f"{i}: {c}")
        
        if already_labeled_image:
            print("Recommended concept:", already_labeled[already_labeled_image][concept])

        print(f"\nPlease select the correct concept:")
        print("- Type an index (0-9) to choose from the above list.")
        print("- Or type your own concept (>= 3 chars).")
        print("- Or type 'accept' to use the recommended concept.")

        __import__("time").sleep(0.5)
        if not labeled_image:
            display(image)  # show image after printing
        if labeled_image:
            __import__("time").sleep(0.5)
            left = Output()
            right = Output()
            with left:
                display(image)
            with right:
                display(labeled_image)
            display(HBox([left, right]))  # show both images side by side

        label = input(f"\nYour selection for {concept}: ").strip()

        try:
            label = int(label)
            if 0 <= label < len(sample['concepts'][concept]):
                concept_labels[image_id][concept] = sample['concepts'][concept][label]
            else:
                raise ValueError
        except:
            if label.lower() == 'accept':
                concept_labels[image_id][concept] = already_labeled[already_labeled_image][concept]
            elif isinstance(label, str) and len(label) > 2:
                concept_labels[image_id][concept] = label
            else:
                print("[Error] Invalid input. Skipping.")
                del concept_labels[image_id]
                break

    # Save after each image
    with open(label_path, "w") as f:
        json.dump(concept_labels, f, indent=2)

    n_samples -= 1
    print(f"[✓] Saved labels for {image_id}\n")

In [53]:
concept_labels = json.load(open(f'checkpoints/{experiment}/{subset}_round{step}/cub200/concept_labels.json', "r"))
len(concept_labels)

110

In [54]:
with open(f'checkpoints/{experiment}/{subset}_round{step+1}/cub200/concept_labels.json', "w") as f:
    json.dump(concept_labels, f, indent=2)

In [55]:
# original_data = pd.read_csv(f'checkpoints/{experiment}/phase1/cub200/data_train.csv')
# len(original_data)

In [56]:
# next_data = pd.read_csv(f'checkpoints/{experiment}/{subset}_round{step+1}/cub200/data_train.csv')
# len(next_data)

In [57]:
# filter next_data to exclude images that have been labeled
# next_images = next_data[~next_data['image_file'].isin(concept_labels.keys())]['image_file'].to_list()
# len(next_images)

In [58]:
# next_images += list(concept_labels.keys())
# len(next_images)

In [59]:
# next_data = original_data[original_data['image_file'].isin(next_images)]
# len(next_data)

In [60]:
# next_data.to_csv(f'checkpoints/{experiment}/{subset}_round{step+1}/cub200/data_train.csv', index=False)
# print(f"[✓] Updated training data for round {step+1} with {len(concept_labels)} labeled images.\n")

#### 